# Efficient Usage of Different Models

The following code suppresses warning messages to ensure the output is clean and focused on the essential results.

In [1]:
import warnings
import os
warnings.filterwarnings('ignore')
os.environ['PIP_ROOT_USER_ACTION'] = 'ignore'

Imports essential modules from CrewAI and initializes two tools, one for web scraping and one for search engine interaction. Also, sets up environment variables to securely store API keys required for accessing external services.

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from crewai import Agent, Task
import os
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

# Create a search tool
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

Initialize different models for different agents.

In [15]:
from crewai import LLM

## call the gemini models
gemini=ChatGoogleGenerativeAI(model="gemini-1.5-flash",
                           verbose=True,
                           temperature=0.5,
                           google_api_key=os.getenv("GOOGLE_API_KEY"))
# Initialize the GPT-4 model using ChatOpenAI
gpt=ChatOpenAI(model="gpt-4o-2024-08-06",
               verbose=True,
               temperature=0.5,
               openai_api_key=os.getenv("OPENAI_API_KEY"))

gemini_llm=LLM(
    model="gemini/gemini-2.5-flash",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.2
)

chatgpt_llm=LLM(
    model="gpt-4o-2024-08-06",
    api_key=os.getenv("OPEN_API_KEY"),
    temperature=0.2
)

Defines two agents, each leveraging a different LLM.

In [ ]:
from crewai import Agent

# Data Researcher Agent using Gemini and SerperSearch
article_researcher=Agent(
    role="Senior Researcher",
    goal='Unccover ground breaking technologies in {topic}',
    verbose=True,
    memory=True,
    backstory=(
        "Driven by curiosity, you're at the forefront of"
        "innovation, eager to explore and share knowledge that could change"
        "the world."
    ),
    tools=[search_tool],
    llm=gemini_llm,
    allow_delegation=True
)

# Article Writer Agent using GPT
article_writer = Agent(
  role='Writer',
  goal='Narrate compelling tech stories about topic : {topic}',
  verbose=True,
  memory=True,
  backstory=(
    "With a flair for simplifying complex topics, you craft"
    "engaging narratives that captivate and educate, bringing new"
    "discoveries to light in an accessible manner."
  ),
  tools=[search_tool],
  llm=chatgpt_llm,
  allow_delegation=False
)

Defining the tasks

In [ ]:
# Research Task
research_task = Task(
    description=(
        "Conduct a thorough analysis on the given topic : {topic}."
        "Utilize SerperSearch for any necessary online research. "
        "Summarize key findings in a detailed report."
    ),
    expected_output='A detailed report on the data analysis with key insights.',
    tools=[search_tool],
    agent=article_researcher,
)

# Writing Task
writing_task = Task(
    description=(
        "Write an insightful article based on the data analysis report. "
        "The article should be clear, engaging, and easy to understand."
    ),
    expected_output='A 6-paragraph article summarizing the data insights.',
    agent=article_writer,
)

Creates a crew and then kicks off the project.

In [19]:
from crewai import Crew, Process

# Form the crew and define the process
crew = Crew(
    agents=[article_researcher, article_writer],
    tasks=[research_task, writing_task],
    process=Process.sequential
)

research_inputs = {
    'topic': 'The rise in Quant Roles in India'
}

# Kick off the crew
result = crew.kickoff(inputs=research_inputs)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Task: Conduct a thorough analysis on the given The rise in Quant Roles in India.Utilize SerperSearch for any   │
│  necessary online research. Summarize key findings in a detailed report.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'The rise in Quant Roles in India', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Quant finance boom in India: intern salaries, global firms, and ...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The rise of Quant Roles in India is a significant trend driven by several factors, positioning India as a      │
│  global hub for quantitative finance and high-frequency trading. This detailed report analyzes the key          │
│  drivers, opportunities, and challenges associated with this burgeoning sector.                                 │
│                                                                                                                 │
│  **Key Findings:**                                                                                              │
│                                                                                                                 │
│  **1. Derivatives Boom and Market Growth:**                                                                     │
│  *   India's derivatives market is experiencing substantial growth, creating a strong demand for quantitative   │
│  skills. This boom is a primary catalyst for the increased need for quants who can develop and implement        │
│  complex trading strategies, risk models, and pricing algorithms.                                               │
│  *   The global quant fund sector is also projected for significant expansion, further fueling the demand for   │
│  talent in India.                                                                                               │
│                                                                                                                 │
│  **2. Talent Pool and Tech-Savvy Workforce:**                                                                   │
│  *   India possesses a large and growing pool of tech-savvy talent, particularly in engineering, mathematics,   │
│  and computer science, which are foundational for quantitative finance. This readily available skilled          │
│  workforce makes India an attractive destination for global quant firms.                                        │
│  *   Educational institutions are also responding to this demand, with programs like IIM Ahmedabad's "Advanced  │
│  Programme in Quantitative Finance & Risk Management" and certifications like the Certificate in Quantitative   │
│  Finance (CQF) gaining prominence as key credentials.                                                           │
│                                                                                                                 │
│  **3. Global Firms and Investment:**                                                                            │
│  *   Major global quant firms are increasingly flocking to India, establishing or expanding their operations.   │
│  This influx of international players signifies confidence in India's potential as a quant finance hub.         │
│  *   These firms are eyeing both the talent pool and the growing trade volume in India, indicating a long-term  │
│  strategic interest.                                                                                            │
│                                                                                                                 │
│  **4. High Salaries and Career Prospects:**                                                                     │
│  *   Quant roles in India offer highly competitive salaries, with interns in top firms earning over $14,000     │
│  USD per month (approximately ₹1.23 million).          

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Task: Write an insightful article based on the data analysis report. The article should be clear, engaging,    │
│  and easy to understand.                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Writer                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **The Rise of Quant Roles in India: A New Frontier in Finance**                                                │
│                                                                                                                 │
│  In recent years, India has witnessed a remarkable surge in the demand for quantitative finance roles,          │
│  positioning itself as a burgeoning hub for this specialized sector. This trend is primarily driven by the      │
│  explosive growth of the derivatives market, which has created a robust demand for professionals skilled in     │
│  developing complex trading strategies, risk models, and pricing algorithms. As the global quant fund sector    │
│  continues to expand, India stands out as a key player, attracting both domestic and international attention.   │
│                                                                                                                 │
│  India's advantage lies in its vast pool of tech-savvy talent, particularly in fields such as engineering,      │
│  mathematics, and computer science. These disciplines form the backbone of quantitative finance, and India's    │
│  educational institutions are rising to the occasion by offering specialized programs. Notable among these are  │
│  IIM Ahmedabad's "Advanced Programme in Quantitative Finance & Risk Management" and the Certificate in          │
│  Quantitative Finance (CQF), which are gaining recognition as essential credentials for aspiring quants.        │
│                                                                                                                 │
│  The influx of major global quant firms into India further underscores the country's potential as a quant       │
│  finance hub. These firms are not only drawn by the skilled workforce but also by India's growing trade         │
│  volume, indicating a long-term strategic interest. This trend signifies a vote of confidence in India's        │
│  capabilities and its role in the global financial landscape. The presence of these international players is    │
│  expected to enhance the quality and scope of work available to Indian quants.                                  │
│                                                                                                                 │
│  One of the most attractive aspects of quant roles in India is the lucrative compensation packages on offer.    │
│  Interns at top firms can earn over $14,000 USD per month, while experienced quantitative analysts can command  │
│  salaries ranging from ₹11,50,000 to ₹27,00,000 annually. This competitive pay scale is a significant draw for  │
│  top talent, further fueling the growth of the sector. The projected 6.0% growth in the job market for          │
│  financial quantitative analysts in the United States between 2022 and 2032 reflects a global trend that India  │
│  is well-positioned to capitalize on.                                                                           │
│                                                                                                                 │
│  Despite the promising opportunities, the quant sector in India is not without its challenges. Discussions on   │
│  platforms like Reddit highlight concerns about the quality of work in certain Global Capability Center (GCC)   │
│  setups, where some Indian quants may face management i

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Converts the crew’s output into a Markdown string for better readability and displays the results in a formatted manner.

In [20]:
from IPython.display import Markdown

# Convert the CrewOutput object to a Markdown string
result_markdown = result.raw

# Display the result as Markdown
Markdown(result_markdown)

**The Rise of Quant Roles in India: A New Frontier in Finance**

In recent years, India has witnessed a remarkable surge in the demand for quantitative finance roles, positioning itself as a burgeoning hub for this specialized sector. This trend is primarily driven by the explosive growth of the derivatives market, which has created a robust demand for professionals skilled in developing complex trading strategies, risk models, and pricing algorithms. As the global quant fund sector continues to expand, India stands out as a key player, attracting both domestic and international attention.

India's advantage lies in its vast pool of tech-savvy talent, particularly in fields such as engineering, mathematics, and computer science. These disciplines form the backbone of quantitative finance, and India's educational institutions are rising to the occasion by offering specialized programs. Notable among these are IIM Ahmedabad's "Advanced Programme in Quantitative Finance & Risk Management" and the Certificate in Quantitative Finance (CQF), which are gaining recognition as essential credentials for aspiring quants.

The influx of major global quant firms into India further underscores the country's potential as a quant finance hub. These firms are not only drawn by the skilled workforce but also by India's growing trade volume, indicating a long-term strategic interest. This trend signifies a vote of confidence in India's capabilities and its role in the global financial landscape. The presence of these international players is expected to enhance the quality and scope of work available to Indian quants.

One of the most attractive aspects of quant roles in India is the lucrative compensation packages on offer. Interns at top firms can earn over $14,000 USD per month, while experienced quantitative analysts can command salaries ranging from ₹11,50,000 to ₹27,00,000 annually. This competitive pay scale is a significant draw for top talent, further fueling the growth of the sector. The projected 6.0% growth in the job market for financial quantitative analysts in the United States between 2022 and 2032 reflects a global trend that India is well-positioned to capitalize on.

Despite the promising opportunities, the quant sector in India is not without its challenges. Discussions on platforms like Reddit highlight concerns about the quality of work in certain Global Capability Center (GCC) setups, where some Indian quants may face management issues or lack access to high-quality projects. Additionally, the complexities of quant investing in India require a nuanced understanding of the local market, suggesting that success in this field demands more than just technical expertise.

In conclusion, India's emergence as a critical player in the global quantitative finance landscape is driven by a confluence of factors: a booming derivatives market, a skilled talent pool, increasing investment from global firms, and attractive compensation packages. While challenges related to work quality and market complexities exist, the overall trajectory points towards sustained growth and expanding opportunities for quantitative professionals in India. The emphasis on specialized skills and certifications highlights the evolving and sophisticated nature of this sector, making it an exciting frontier for finance professionals.